In [0]:
USE CATALOG dbacademy;
USE SCHEMA labuser14586003_1778076074;

In [0]:
%python
spark.sql("""
SELECT * ,
      upper(customer_name) as Customer_Name_Upper,
      date(current_timestamp()) as processDate
FROM bronze_table""").createOrReplaceTempView("silver_source")

In [0]:
%sql
SELECT * FROM silver_source

### **MERGE Using PySpark**

In [0]:
%python
if spark.catalog.tableExists('silver_table'):
    pass


else:
    spark.sql("""
                CREATE TABLE IF NOT EXISTS silver_table
                AS 
                SELECT * FROM silver_source""")

### **MERGE USING SQL**

In [0]:
%sql
CREATE TABLE IF NOT EXISTS silver_table
                AS 
                SELECT * FROM silver_source

In [0]:
%sql
MERGE INTO silver_table
USING silver_source
ON silver_table.order_id = silver_source.order_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

In [0]:
%sql
SELECT * FROM silver_table

In [0]:
SELECT *,
row_number() OVER(ORDER BY customer_id) AS DimCustomerskey
FROM (
SELECT DISTINCT customer_id, customer_name, customer_email
FROM silver_table
) AS T